In [1]:

import pandas as pd
import numpy as np

df4 = pd.read_csv("/content/sentiment_reviews.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df4.shape)
print("\nSentiment counts:")
print(df4["sentiment"].value_counts())

display(df4.head())


Dataset loaded successfully!
Dataset shape: (10, 3)

Sentiment counts:
sentiment
positive    5
negative    4
neutral     1
Name: count, dtype: int64


,review_id,review_text,sentiment
0,1,"The product quality is exceptional, exceeded a...",positive
1,2,Terrible customer service and the item arrived...,negative
2,3,"Average performance, decent for the price poin...",neutral
3,4,"Absolutely love this! Easy to setup, robust, a...",positive
4,5,Stopped working after two days. Extremely disa...,negative


In [2]:

import re
import html

def clean_text(text):
    text = html.unescape(str(text))
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df4 = df4.dropna(subset=["review_text", "sentiment"]).copy()
df4["clean_review"] = df4["review_text"].apply(clean_text)
df4["sentiment"] = df4["sentiment"].str.strip().str.lower()

print("Text cleaning completed!")
display(df4[["review_text", "clean_review", "sentiment"]].head())


Text cleaning completed!


,review_text,clean_review,sentiment
0,"The product quality is exceptional, exceeded a...",the product quality is exceptional exceeded al...,positive
1,Terrible customer service and the item arrived...,terrible customer service and the item arrived...,negative
2,"Average performance, decent for the price poin...",average performance decent for the price point...,neutral
3,"Absolutely love this! Easy to setup, robust, a...",absolutely love this easy to setup robust and ...,positive
4,Stopped working after two days. Extremely disa...,stopped working after two days extremely disap...,negative


In [3]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

X_text = df4["clean_review"]
y = df4["sentiment"]

models4 = {
    "Multinomial Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2)
        )),
        ("classifier", MultinomialNB())
    ]),
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2)
        )),
        ("classifier", LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        ))
    ])
}

for name, model in models4.items():
    model.fit(X_text, y)
    print(name, "trained successfully!")


Multinomial Naive Bayes trained successfully!
Logistic Regression trained successfully!


In [4]:

def predict_sentiment(review):
    cleaned = clean_text(review)
    print("Review:", review)

    for name, model in models4.items():
        prediction = model.predict([cleaned])[0]
        print(f"{name}: {prediction}")

predict_sentiment("The product is excellent and I love it!")
predict_sentiment("Very poor quality and terrible service.")
predict_sentiment("The product is okay, nothing special.")


Review: The product is excellent and I love it!
Multinomial Naive Bayes: positive
Logistic Regression: positive
Review: Very poor quality and terrible service.
Multinomial Naive Bayes: positive
Logistic Regression: negative
Review: The product is okay, nothing special.
Multinomial Naive Bayes: positive
Logistic Regression: positive


In [5]:

from sklearn.metrics import classification_report, accuracy_score

for name, model in models4.items():
    predictions = model.predict(X_text)

    print("\n", name)
    print("Training accuracy:",
          round(accuracy_score(y, predictions) * 100, 2), "%")

    print(classification_report(
        y, predictions,
        labels=["negative", "neutral", "positive"],
        zero_division=0
    ))



 Multinomial Naive Bayes
Training accuracy: 90.0 %
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00         4
     neutral       0.00      0.00      0.00         1
    positive       0.83      1.00      0.91         5

    accuracy                           0.90        10
   macro avg       0.61      0.67      0.64        10
weighted avg       0.82      0.90      0.85        10


 Logistic Regression
Training accuracy: 100.0 %
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00         4
     neutral       1.00      1.00      1.00         1
    positive       1.00      1.00      1.00         5

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10



In [6]:

results4 = []

for name, model in models4.items():
    pred = model.predict(X_text)

    results4.append({
        "Model": name,
        "Training Accuracy": accuracy_score(y, pred),
        "Number of Reviews": len(y)
    })

results4_df = pd.DataFrame(results4)
display(results4_df)

results4_df.to_csv(
    "task4_sentiment_results.csv", index=False
)

df4.to_csv("task4_cleaned_reviews.csv", index=False)

print("Task 4 result files saved!")


,Model,Training Accuracy,Number of Reviews
0,Multinomial Naive Bayes,0.9,10
1,Logistic Regression,1.0,10


Task 4 result files saved!
